# Sprint 3 - Week 2 - Class 1

Valores Duplicados, Ausentes y Filtrado

En este caso de estudio, vamos a analizar el comportamiento del ranking
FIFA desde que inició su medición en el año 1993, a fin de presentar al
público la situación y el rendimiento de las distintas selecciones
nacionales afiliadas.

En este contexto, nos interesa contestar entre otras las siguientes
preguntas:

-   ¿Cómo es el rendimiento histórico por confederación?
-   ¿Cómo ha sido el rendimiento histórico de su país comparado con el
    promedio de la confederación a la cual pertenece?
-   ¿Cuáles han sido históricamente los mejores países?
-   ¿Quienes han sido los top 5 países previo al inicio de cada mundial?
    ¿Entre estos países han estado los campeones del mundo
    correspondientes?
-   Según esta puntuación, ¿las selecciones son cada vez más
    competitivas o cada vez parece existir más diferencia a través del
    tiempo?

Con este propósito usted cuenta con un conjunto de datos en el archivo
`fifa_rank.csv` cuya metadata se detalla a continuación:

-   Fuente: FIFA Site 2018
-   Dimensiones:
    -   `Country`: Nombre del pais afiliado a la FIFA cuya selección de
        fútbol es puntuada
    -   `Confederation`: Confederación a la que pertenece de acuerdo a
        la división de la FIFA
    -   `Rank_Date`: Fecha de publicación del ranking FIFA entre marzo
        de 1993 y junio de 2018 (previo al mundial de Rusia)
    -   `Points_Old_Version`: Puntos obtenidos conforme el sistema
        antiguo de la FIFA (previo al año 2011)
    -   `Ponts_New_Version`: Puntos obtenidos conforme el sistema nuevo
        de la FIFA (a partir del años 2011)
    -   `rank`: Posición en el ranking oficial de la FIFA dados los
        puntos calculados

## CARGAR LIBRERIAS Y DATOS

In [1]:
#Cargar librerias
import pandas as pd
import numpy as np
pd.options.display.max_columns = None

### Ajustar ruta relativa según contexto

In [2]:
#Cargar datos
df_fifa = pd.read_csv("fifa_rank.csv", sep = ",")

## DIAGNÓSTICO INICIAL

In [3]:
#Visualizar información general de los datos
df_fifa.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 57797 entries, 0 to 57796
Data columns (total 6 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Country             57797 non-null  object 
 1   Confederation       54815 non-null  object 
 2   Rank_Date           57797 non-null  object 
 3   Points_Old_Version  40382 non-null  float64
 4   Points_New_Version  57797 non-null  object 
 5   rank                57797 non-null  int64  
dtypes: float64(1), int64(1), object(4)
memory usage: 2.6+ MB


In [4]:
#Visualizar cabecera de los datos
df_fifa.head(n = 10)

,Country,Confederation,Rank_Date,Points_Old_Version,Points_New_Version,rank
0,Germany,UEFA,1993-08-08,57.0,ND,1
1,Italy,UEFA,1993-08-08,57.0,ND,2
2,Switzerland,UEFA,1993-08-08,50.0,ND,3
3,Sweden,UEFA,1993-08-08,55.0,ND,4
4,Argentina,CONMEBOL,1993-08-08,51.0,ND,5
5,Republic of Ireland,UEFA,1993-08-08,54.0,ND,6
6,Russia,UEFA,1993-08-08,52.0,ND,7
7,Brazil,CONMEBOL,1993-08-08,55.0,ND,8
8,Norway,UEFA,1993-08-08,49.0,ND,9
9,Denmark,UEFA,1993-08-08,51.0,ND,10


In [5]:
#Visualizar una muestra aleatoria de los datos
df_fifa.sample(n = 10)

,Country,Confederation,Rank_Date,Points_Old_Version,Points_New_Version,rank
5226,Turkmenistan,AFC,1996-07-03,12.0,ND,141
36941,Panama,CONCACAF,2010-03-03,406.0,ND,78
40977,Comoros,CAF,2011-10-19,NaN,72.48,183
12314,Iceland,UEFA,1999-12-22,533.0,ND,43
42074,USA,CONCACAF,2012-04-11,NaN,778.69,29
12379,Uganda,CAF,1999-12-22,342.0,ND,108
54504,Qatar,AFC,2017-03-09,NaN,415.12,84
57618,Venezuela,CONMEBOL,2018-06-07,NaN,754.55,33
52579,Serbia,UEFA,2016-06-02,NaN,575.85,54
54686,Slovenia,UEFA,2017-04-06,NaN,614.29,55


## PROCESAMIENTO

### Ajustar nombres de columnas

-   Poner todos los nombres de columnas en minusculas

In [6]:
col_originales = df_fifa.columns
col_nuevas = [col.lower() for col in col_originales]
df_fifa.columns = col_nuevas

In [7]:
df_fifa.head(10)

,country,confederation,rank_date,points_old_version,points_new_version,rank
0,Germany,UEFA,1993-08-08,57.0,ND,1
1,Italy,UEFA,1993-08-08,57.0,ND,2
2,Switzerland,UEFA,1993-08-08,50.0,ND,3
3,Sweden,UEFA,1993-08-08,55.0,ND,4
4,Argentina,CONMEBOL,1993-08-08,51.0,ND,5
5,Republic of Ireland,UEFA,1993-08-08,54.0,ND,6
6,Russia,UEFA,1993-08-08,52.0,ND,7
7,Brazil,CONMEBOL,1993-08-08,55.0,ND,8
8,Norway,UEFA,1993-08-08,49.0,ND,9
9,Denmark,UEFA,1993-08-08,51.0,ND,10


### Visualizar datos en confederation

-   Visualizar cuantos casos hay por confederation

In [9]:
#Visualizar cuantos casos por confederation
df_fifa.groupby('confederation', dropna=False)['country'].count()

confederation
AFC         12481
CAF         14876
CONCACAF     9664
CONMEBOL     2860
UEFA        14934
NaN          2982
Name: country, dtype: int64

### Ajustar formato de columna rank_date

-   Cambiar a formato fecha

In [10]:
df_fifa['rank_date'] = pd.to_datetime(df_fifa['rank_date'], infer_datetime_format=True)

C:\Users\dell-inspiron15\AppData\Local\Temp\ipykernel_12184\3960985096.py:1: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  df_fifa['rank_date'] = pd.to_datetime(df_fifa['rank_date'], infer_datetime_format=True)


In [11]:
df_fifa.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 57797 entries, 0 to 57796
Data columns (total 6 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   country             57797 non-null  object        
 1   confederation       54815 non-null  object        
 2   rank_date           57797 non-null  datetime64[ns]
 3   points_old_version  40382 non-null  float64       
 4   points_new_version  57797 non-null  object        
 5   rank                57797 non-null  int64         
dtypes: datetime64[ns](1), float64(1), int64(1), object(3)
memory usage: 2.6+ MB


### Adicionar columnas de año y mes

-   Crear campo `year`
-   Crear campo `month`
-   Cuantos casos hay por cada mes y año
-   Hacer un primer grafico considerando que existen demasiados casos

In [12]:
#Campo year
df_fifa['year'] = pd.DatetimeIndex(df_fifa['rank_date']).year

#Campo month
df_fifa['month'] = pd.DatetimeIndex(df_fifa['rank_date']).month

In [13]:
df_fifa.sample(10)

,country,confederation,rank_date,points_old_version,points_new_version,rank,year,month
11920,Slovenia,UEFA,1999-10-13,522.0,ND,53,1999,10
42012,Dominica,CONCACAF,2012-03-07,NaN,95.43,174,2012,3
50717,Burkina Faso,CAF,2015-09-03,NaN,467.73,73,2015,9
3371,Sudan,CAF,1995-07-25,22.0,ND,91,1995,7
6286,Mozambique,CAF,1997-02-27,32.0,ND,83,1997,2
2146,Ethiopia,CAF,1994-10-25,12.0,ND,115,1994,10
46237,Somalia,CAF,2013-11-28,NaN,5.99,204,2013,11
16713,Aruba,CONCACAF,2001-09-19,101.0,ND,186,2001,9
10956,Singapore,AFC,1999-05-19,367.0,ND,99,1999,5
29422,Cameroon,CAF,2007-02-14,1160.0,ND,17,2007,2


In [14]:
#Visualizar cuantos casos hay por cada mes y año
df_fifa.groupby(['year','month'])['country'].count()

year  month
1993  8        167
      9        167
      10       167
      11       168
      12       168
              ... 
2018  2        211
      3        211
      4        211
      5        211
      6        211
Name: country, Length: 285, dtype: int64

In [15]:
#Hacer un primer grafico considerando que existen demasiados casos
df_grupos_meses = df_fifa.groupby(['year','month'])['country'].count()
df_grupos_meses.plot(
    kind="line",
    xlabel="Año, Mes",
    ylabel="# Paises",
    title="Cantidad de Paises Calificados"
)

ImportError: matplotlib is required for plotting when the default backend "matplotlib" is selected.

### Visualizar datos en Points_Old_Version

-   Visualizar cuantos casos hay para cada valor posibles
-   Visualizar cuantos NA hay por año - mes

In [17]:
#Visualizar cuantos casos
df_fifa['points_old_version'].value_counts(dropna=False)

points_old_version
NaN       17415
0.0        1044
7.0         301
11.0        295
3.0         295
          ...  
1250.0        1
1102.0        1
1047.0        1
1043.0        1
942.0         1
Name: count, Length: 1376, dtype: int64

In [18]:
df_fifa[df_fifa['points_old_version'].isna()][['year','month']].value_counts().sort_index()

year  month
2011  8        206
      9        207
      10       207
      11       208
      12       209
              ... 
2018  2        211
      3        211
      4        211
      5        211
      6        211
Name: count, Length: 83, dtype: int64

### Visualizar datos en Points_New_Version

-   Visualizar cuantos casos hay para cada valor posibles
-   Visualizar cuantos “no definidos” (ND) hay por año

In [19]:
#Visualizar cuantos casos
df_fifa['points_new_version'].value_counts(dropna=False)

points_new_version
ND         40382
0            333
63.75         48
38.25         39
66            33
           ...  
1156.01        1
1174.34        1
1176.88        1
1329.86        1
165.84         1
Name: count, Length: 11115, dtype: int64

In [18]:
df_fifa[df_fifa['points_new_version']== "ND"]['year'].value_counts().sort_index()

year
1993     837
1994    1728
1995    1792
1996    1847
1997    1916
1998    1938
1999    2416
2000    2429
2001    2437
2002    2233
2003    2450
2004    2456
2005    2461
2006    2259
2007    2489
2008    2485
2009    2484
2010    2277
2011    1448
Name: count, dtype: int64

## PROCESAMIENTO 2

### E1: Ajustar valores perdidos para variables que correspondan

-   Variable confederation
-   Variable poins_new_version

In [39]:
# 

df_fifa['confederation']= df_fifa['confederation'].fillna('OTHER')
df_fifa['confederation'].value_counts(dropna=False)

df_fifa.loc[df_fifa['points_new_version'] == 'ND', 'points_new_version'] = np.nan
df_fifa['points_new_version'] = df_fifa['points_new_version'].astype(float)
df_fifa['points_new_version'].value_counts(dropna=False)
df_fifa.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 57797 entries, 0 to 57796
Data columns (total 8 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   country             57797 non-null  object        
 1   confederation       57797 non-null  object        
 2   rank_date           57797 non-null  datetime64[ns]
 3   points_old_version  40382 non-null  float64       
 4   points_new_version  17415 non-null  float64       
 5   rank                57797 non-null  int64         
 6   year                57797 non-null  int32         
 7   month               57797 non-null  int32         
dtypes: datetime64[ns](1), float64(2), int32(2), int64(1), object(2)
memory usage: 3.1+ MB


### E2: Encontrar y dar tratamiento a duplicados globales

-   Identificar duplicados globales
-   Eliminar duplicados globales

In [52]:
# 
df_fifa=df_fifa.drop_duplicates()
df_fifa[df_fifa.duplicated()]

,country,confederation,rank_date,points_old_version,points_new_version,rank,year,month


### E3: Encontrar y dar tratamiento a duplicados implícitos

-   Estudiar duplicados implicitos por año y mes
-   Estudiar algunos casos específicos de duplicados implícitos y
    analizar qué podría estar pasando.
-   Eliminar duplicados implicitos
-   Volver a graficar serie de tiempo

In [53]:
# 
df_fifa[df_fifa.duplicated(subset=['year','month'])]

,country,confederation,rank_date,points_old_version,points_new_version,rank,year,month
1,Italy,UEFA,1993-08-08,57.0,NaN,2,1993,8
2,Switzerland,UEFA,1993-08-08,50.0,NaN,3,1993,8
3,Sweden,UEFA,1993-08-08,55.0,NaN,4,1993,8
4,Argentina,CONMEBOL,1993-08-08,51.0,NaN,5,1993,8
5,Republic of Ireland,UEFA,1993-08-08,54.0,NaN,6,1993,8
...,...,...,...,...,...,...,...,...
57792,Anguilla,CONCACAF,2018-06-07,NaN,0.0,206,2018,6
57793,Bahamas,CONCACAF,2018-06-07,NaN,0.0,206,2018,6
57794,Eritrea,CAF,2018-06-07,NaN,0.0,206,2018,6
57795,Somalia,CAF,2018-06-07,NaN,0.0,206,2018,6


### E4: Crear puntuacion unificada

-   Crear nueva variable points que junte las columnas de puntos
    antiguos y nuevos
-   Quitar variables de puntuacion antigua y nueva

In [36]:
# 